# 📋 DAT09 · 미션10 전체 실행을 검증하고 담당자에게 인계하기 (학습자용)

---

## 실습 시간·작업 폴더·연결 파일

- **예상 실습 시간**: 3시간 (전체 10개 미션 총 27시간 과정)
- **작업 폴더**: `DAT09_practice`
- **연결 파일**: 첫 셀에서 `DAT09.md`를 연결하세요.

## 이전 미션 연계와 실행 결과

- **이전 미션 연계**: 같은 작업 폴더에서 미션1~미션9을 먼저 실행했다면, 첫 셀이 `src/m01.py`~`src/m09.py`를 불러와 작업 사전 `c`를 자동으로 복원합니다.

- 각 단계 함수(`m10_s1`~`m10_s5`)의 `TODO`를 직접 작성한 뒤 셀을 실행하세요.
- 실행 결과는 `outputs/M10/DAT09_M10_result.html`과 `step_01.csv`~`step_05.csv`에 저장됩니다.



In [ ]:
from pathlib import Path, PurePosixPath
import sys, io, json, re, zipfile, base64, hashlib, importlib
from IPython.display import display, HTML
import html
import pandas as pd
ROLE = 'student'
EXPECTED_SHA256 = 'a0adfd7b9a2416cd74af2fc99bb7978ad662eec8d9e40ce5b8a539977c2cb17e'
c = None

def connect_md(path):
    global c, ROOT, HERE
    path = Path(path)
    if path.name not in ('DAT09.md', 'DAT09.md'):
        raise ValueError('원자료 DAT09.md 또는 DAT09.md를 선택하세요. 정답코드 MD는 연결 파일이 아닙니다.')
    text = path.read_text(encoding='utf-8-sig')
    found = re.search(r'<!-- DAT09_RUNTIME_BUNDLE_B64:([A-Za-z0-9+/=]+) -->', text)
    if not found:
        raise ValueError('이 노트북용 원자료 블록이 없습니다. 함께 받은 DAT09.md를 사용하세요.')
    raw = base64.b64decode(found.group(1), validate=True)
    if hashlib.sha256(raw).hexdigest() != EXPECTED_SHA256:
        raise ValueError('자료와 노트북 버전이 다릅니다. 같은 묶음의 파일을 사용하세요.')
    ROOT = Path.cwd() / ('DAT09_reference' if ROLE == 'solution' else 'DAT09_practice')
    ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        for info in archive.infolist():
            name = PurePosixPath(info.filename)
            if name.is_absolute() or '..' in name.parts:
                raise ValueError('허용되지 않은 경로입니다.')
            target = ROOT.joinpath(*name.parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(info))
    sys.path.insert(0, str(ROOT))
    for name in list(sys.modules):
        if name in ['dat09_common', 'dat09_ui'] or name.startswith('src.m'):
            del sys.modules[name]
    import dat09_common
    globals().update({name: getattr(dat09_common, name) for name in dir(dat09_common) if not name.startswith('_')})
    import collections, itertools, queue, threading, tempfile, importlib.util, shutil
    globals().update({k:v for k,v in locals().items() if k in ['collections','itertools','re','queue','threading','tempfile','importlib','shutil']})
    from dat09_ui import bars, heatmap, line_chart, dashboard_fragment
    globals().update(bars=bars,heatmap=heatmap,line_chart=line_chart,dashboard_fragment=dashboard_fragment)
    c = dat09_common.context(ROOT)
    HERE = ROOT.parent / 'instructor'
    for candidate in [Path.cwd()/'instructor', Path.cwd(), ROOT.parent/'instructor']:
        if (candidate/'aar_key.json').is_file():
            HERE = candidate
            c['instructor_root'] = candidate
            break
    print('자료 연결 완료:', ROOT)
    print('10개 미션·50개 함수·27시간 과정입니다. 미션1부터 실행하세요.')
    print('TODO가 남아 있으면 해당 단계에서 멈춥니다. src에는 작성한 함수를 저장합니다.')
    return c

def hydrate_previous_missions(c, up_to=9):
    """같은 작업 폴더에 저장된 이전 미션(src/m01.py~)을 순차 실행해 작업 사전 c를 복원합니다."""
    import importlib.util
    src_dir = c['root'] / 'src'
    missing = [k for k in range(1, up_to + 1) if not (src_dir / f'm{k:02d}.py').is_file()]
    if missing:
        miss_str = ', '.join(f'미션{k}(DAT09_m{k:02d}_*.ipynb)' for k in missing)
        print(f'[안내] 아직 완료되지 않은 이전 미션이 있습니다: {miss_str}')
        print('이전 미션 노트북을 같은 폴더에서 먼저 실행한 뒤 이 셀을 다시 실행하세요.')
        return False
    for k in range(1, up_to + 1):
        if k in c['completed']:
            continue
        mod_path = src_dir / f'm{k:02d}.py'
        spec = importlib.util.spec_from_file_location(f'dat09_prev_m{k:02d}', mod_path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        fns = [getattr(mod, f'm{k:02d}_s{s}') for s in range(1, 6)]
        run_mission(c, k, fns)
    print(f'이전 미션(미션1~미션{up_to}) 상태 복원 완료.')
    return True

DATA_MD = next((Path(p) for p in ('DAT09.md', 'DAT09.md') if Path(p).is_file()), None)
if DATA_MD is not None:
    connect_md(DATA_MD)
    hydrate_previous_missions(c, 9)
else:
    print('DAT09.md(또는 DAT09.md)를 이 노트북과 같은 폴더에 두고 첫 셀을 다시 실행하세요.')
    print("다른 경로에 있다면 connect_md('/자료가_있는_경로/DAT09.md'); hydrate_previous_missions(c)를 실행하세요.")


## 미션10 전체 실행을 검증하고 담당자에게 인계하기

예상 3시간입니다.

#### 이번 미션의 목표

새 프로세스에서 전체 도구를 재실행하고, 원자료부터 화면까지 값을 추적해 다른 담당자가 이어서 사용할 수 있게 인계합니다.

### 1단계 빈 결과 폴더에서 전체를 한 명령으로 실행하기

#### 이 단계에서 하는 일

현재 노트북에 남아 있는 변수나 예전 결과 파일에 의존하면 다른 담당자는 같은 결과를 만들 수 없습니다. 저장된 코드와 입력만으로 새 Python 프로세스에서 처음부터 실행되는지 확인합니다.

#### 시작 전 확인

미션9까지 완료한 결과를 사용합니다. 노트북의 m10_s1(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

미션1~미션9와 현재 작성한 m10 함수, 저장된 src/m01.py~m10.py, data·config·공통 코드를 사용합니다. 새 환경은 새 Python 프로세스와 빈 결과 폴더를 뜻하며 새 OS 설치를 뜻하지 않습니다.

#### 작업 순서

- run_dat09.py가 src의 단계 함수를 읽어 미션1부터 미션10까지 순서대로 실행하게 하세요. 수집·변환·DB·동기화·품질·후보·화면·시험이 실제로 이어져야 합니다.

- --output 인자로 매번 새 결과 폴더를 받게 하고, 부모 프로세스의 c를 전달하지 말고 context를 새로 만드세요.

- 명령 `python run_dat09.py --output 새_폴더`를 새 프로세스로 실행하세요. 미션10 안에서 다시 새 프로세스를 무한히 만드는 일이 없도록 자식 실행에서는 재귀 검증을 생략합니다.

- 종료 코드, 전체 로그와 완료 미션 목록을 저장하세요. 실패하면 마지막 성공 미션·예외·이유를 run_failure.json에 남기고 완료로 표시하지 않습니다.


#### 저장할 결과

작업 루트의 run_dat09.py, clean_run.log와 새 폴더의 run_summary.json 또는 run_failure.json을 남깁니다.

#### 직접 확인하는 방법

종료 코드 0, 완료 목록 1~10, 새 결과 폴더의 열 미션 결과를 확인하세요. 이전 폴더의 파일이 있다는 이유로 성공 처리하지 않습니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M10/step_01.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M10_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 2단계 다섯 종류의 값을 원파일부터 화면까지 추적하기

#### 이 단계에서 하는 일

최종 차트 값이 맞는지 알려면 원래 값이 어떤 변환과 판정을 거쳤는지 확인해야 합니다. 정상뿐 아니라 단위·결측·충돌·지연도 같은 방식으로 따라가야 합니다.

#### 시작 전 확인

미션10의 1단계까지 작성·확인한 결과를 사용합니다. 노트북의 m10_s2(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

원파일, 미션2 normalized.csv, 미션3 DB·logical_stock.csv, 미션4 후보, 미션7 승인 상태와 미션6 화면을 사용합니다. 대표 ID는 정상 A-0719, BOX 환산 B-0719, 결측 A-0000, 값 충돌 B-0030, 지연 C-0060입니다.

#### 작업 순서

- 다섯 ID를 각각 원파일에서 찾고 원래 수량·단위·시각을 기록하세요. 변환 후 EA·UTC·코드와 검사 상태를 같은 행에 적습니다.

- DB에 저장된 원래 보고가 파일의 내용과 같은지 확인하고, 해당 재고 이력에 연결된 세 출처 ID를 찾으세요.

- 후보 유무·후보 수량·승인 상태와 화면 수량·검토 상태를 비교하세요. 결측의 후보를 원래 수량처럼 적거나, 지연으로 제외된 수량을 0으로 쓰지 않습니다.

- 각 사례에 사용한 코드 매핑·검증 규칙 버전과 대조 결과를 남기고 integrated_dashboard.html에서도 같은 대상을 확인하세요.


#### 저장할 결과

lineage_samples.csv와 integrated_dashboard.html, 다섯 사례의 후보·승인 대조 기록을 남깁니다.

#### 직접 확인하는 방법

다섯 ID가 모두 원파일·변환 보고·DB·화면에 연결돼야 합니다. 기본 실행의 승인 상태는 UNAPPROVED이고 확인 불가 화면값은 N/A입니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M10/step_02.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M10_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 3단계 승인 없는 실행과 세 장애 상황 확인하기

#### 이 단계에서 하는 일

전체 실행을 편하게 만들려고 승인을 자동으로 통과시키거나 장애를 숨기면 인수할 수 없습니다. 기본 실행은 후보를 보류한 상태로 끝나야 하며 장애 때 원자료와 정상 결과가 어떻게 보호되는지 확인해야 합니다.

#### 시작 전 확인

미션10의 2단계까지 작성·확인한 결과를 사용합니다. 노트북의 m10_s3(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

기본 DB·후보 목록, load_fn, 미션4 checkpoint.json과 원파일 해시를 사용합니다. 파일 누락·DB 잠김은 별도 시험 경로에서 만들고 실제 기본 파일을 삭제하지 않습니다.

#### 작업 순서

- 기본 승인 마스터가 0건이고 모든 외부 반영이 NOT_CONNECTED인지 조회하세요. 보류·후보 목록이 그대로 남는지 확인합니다.

- 없는 파일 경로를 읽어 수집 실패가 나는지 확인하고 기존 원파일 해시를 비교하세요. 실패한 단계를 성공으로 기록하지 않습니다.

- 임시 DB를 두 연결로 열어 한쪽이 잠근 동안 다른 쪽의 저장 실패를 확인하세요. 재시도가 필요하다는 처리와 기록을 남깁니다. 미션8의 실제 재시도 시험도 함께 제시합니다.

- 체크포인트에서 복원한 처리기에 전체 이벤트를 다시 보내고 재수신 전후 현재값·충돌 목록이 같은지 비교하세요. 시험 범위를 넘는 복구를 했다고 주장하지 않습니다.


#### 저장할 결과

pending_cases.csv, e2e_tests.json과 기본 승인 건수·원파일 해시·장애 로그를 남깁니다.

#### 직접 확인하는 방법

기본 승인 0건, 세 장애 시험 통과를 확인하세요. 원자료 유지 확인과 모든 결과 파일의 원자적 복구는 다른 범위이므로 실제로 확인한 범위를 적습니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M10/step_03.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M10_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 4단계 현재 환경을 다시 평가하고 실행 매뉴얼 작성하기

#### 이 단계에서 하는 일

도구를 만든 사람이 떠나도 다음 담당자가 설치·실행·오류 확인을 할 수 있어야 합니다. 미션1과 같은 기준으로 개선을 확인하되 파일이 생겼다는 이유만으로 모든 기능이 검증됐다고 평가하지 않습니다.

#### 시작 전 확인

미션10의 3단계까지 작성·확인한 결과를 사용합니다. 노트북의 m10_s4(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

미션1 environment_baseline.csv와 실제 결과 파일·시험 로그를 사용합니다. 최종 평가는 파일 연결·코드·품질·적시성·책임·재실행의 같은 여섯 영역을 사용합니다.

#### 작업 순서

- 각 영역을 0점 근거 없음, 1점 문서·수동 확인, 2점 정상·오류를 포함한 시험 확인의 같은 기준으로 다시 평가하세요. 이전·현재 점수와 실제 시험 경로·남은 확인을 적습니다.

- 매뉴얼을 설치, 입력, 코드·단위, 검증·품질, 동기화·복구, 역할별 화면, 승인·감사, 부하·장애, 인수·변경의 아홉 장으로 작성하세요.

- 각 장에 필요한 파일·실행 또는 확인 방법·정상 결과·주요 오류와 조치를 넣으세요. 단순 파일 목록이 아니라 다음 담당자가 따라 할 순서로 씁니다.

- 동료에게 실행 명령 한 개와 오류 조치 한 개를 매뉴얼만 보고 찾게 하고 막힌 부분을 고치세요. 실제 환경 인증·외부 연동·최종 교관 평가는 남은 일로 구별합니다.


#### 저장할 결과

environment_final.csv, final_manual.md와 동료의 매뉴얼 확인 기록을 저장합니다.

#### 직접 확인하는 방법

아홉 장이 모두 있고 실제 파일 경로와 명령이 맞는지 확인하세요. “오류를 해결한다” 대신 확인할 파일과 다시 실행할 조건이 있어야 합니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M10/step_04.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M10_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 5단계 최종 열두 항목을 확인하고 다음 담당자에게 인계하기

#### 이 단계에서 하는 일

마지막 보고는 점수가 높다는 발표가 아니라 무엇을 확인했고 무엇이 남았는지 다음 담당자가 판단할 자료입니다. 코드 시험 통과와 학습자가 이유를 설명할 수 있는지는 별도로 확인합니다.

#### 시작 전 확인

미션10의 4단계까지 작성·확인한 결과를 사용합니다. 노트북의 m10_s5(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

data/acceptance_checks.json의 12항목, 열 미션의 결과·수정 코드·입력·설정과 교관의 미션별 필수 평가 20개를 사용합니다.

#### 작업 순서

- 12항목의 실제 근거를 열어 PASS·FAIL·NOT_RUN을 기록하세요. 자동 코드 점검과 교관 확인을 서로 다른 칸으로 유지합니다. 자동 PASS만으로 학습 합격을 표시하지 않습니다.

- 1쪽 브리핑에 전체·수용·보류·격리 건수, 품질 분자·분모, 값 충돌·비교 불가·승인대기, 실측 성능, 업무시간 가정, 남은 문제를 적으세요.

- handover.md에 다음 담당자·미완료 작업·완료 조건·확인할 파일을 적고 data·config·src·결과·로그·매뉴얼을 함께 묶으세요. 학습자 배포본과 교관 정답 자료는 분리합니다.

- 교관이 바꾼 거점 필터나 허용 별칭 하나에 대해 실행 전에 결과를 예상하고, 재실행 뒤 원자료와 대조해 이유를 설명하세요. 전체 필수 평가와 최종 점검을 마친 뒤 교관이 합격을 판정합니다.


#### 저장할 결과

acceptance_results.json, briefing.md, handover.md와 교관 판정 기록을 저장합니다. 미확인 항목은 확인할 담당·조건을 붙입니다.

#### 직접 확인하는 방법

코드 점검 12/12와 교관 평가 20/20은 다른 결과입니다. 두 종류와 최종 사람 확인이 모두 끝나기 전에는 과정 전체를 합격으로 표시하지 않습니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M10/step_05.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M10_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.



In [ ]:
if c is None:
    raise RuntimeError("첫 셀에서 DAT09.md(또는 DAT09.md)를 먼저 연결하세요.")

# STEP 01 빈 결과 폴더에서 전체를 한 명령으로 실행하기
# 할 일: run_dat09.py가 src의 단계 함수를 읽어 미션1부터 미션10까지 순서대로 실행하게 하세요. 수집·변환·DB·동기화·품질·후보·화면·시험이 실제로 이어져야 합니다. --output 인자로 매번 새 결과 폴더를 받게 하고, 부모 프로세스의 c를 전달하지 말고 context를 새로 만드세요. 명령 `python run_dat09.py --output 새_폴더`를 새 프로세스로 실행하세요. 미션10 안에서 다시 새 프로세스를 무한히 만드는 일이 없도록 자식 실행에서는 재귀 검증을 생략합니다. 종료 코드, 전체 로그와 완료 미션 목록을 저장하세요. 실패하면 마지막 성공 미션·예외·이유를 run_failure.json에 남기고 완료로 표시하지 않습니다.
# 확인: 종료 코드 0, 완료 목록 1~10, 새 결과 폴더의 열 미션 결과를 확인하세요. 이전 폴더의 파일이 있다는 이유로 성공 처리하지 않습니다.
def m10_s1(c):
    """빈 결과 폴더에서 전체를 한 명령으로 실행하기.

    읽을 작업값: root
    저장할 작업값: clean_run_ok
    결과 폴더: folder(c, 10)
    """
    return TODO(10, 1)

# STEP 02 다섯 종류의 값을 원파일부터 화면까지 추적하기
# 할 일: 다섯 ID를 각각 원파일에서 찾고 원래 수량·단위·시각을 기록하세요. 변환 후 EA·UTC·코드와 검사 상태를 같은 행에 적습니다. DB에 저장된 원래 보고가 파일의 내용과 같은지 확인하고, 해당 재고 이력에 연결된 세 출처 ID를 찾으세요. 후보 유무·후보 수량·승인 상태와 화면 수량·검토 상태를 비교하세요. 결측의 후보를 원래 수량처럼 적거나, 지연으로 제외된 수량을 0으로 쓰지 않습니다. 각 사례에 사용한 코드 매핑·검증 규칙 버전과 대조 결과를 남기고 integrated_dashboard.html에서도 같은 대상을 확인하세요.
# 확인: 다섯 ID가 모두 원파일·변환 보고·DB·화면에 연결돼야 합니다. 기본 실행의 승인 상태는 UNAPPROVED이고 확인 불가 화면값은 N/A입니다.
def m10_s2(c):
    """다섯 종류의 값을 원파일부터 화면까지 추적하기.

    읽을 작업값: conn, logical_rows, normalized, root
    저장할 작업값: lineage_tests
    결과 폴더: folder(c, 10)
    """
    return TODO(10, 2)

# STEP 03 승인 없는 실행과 세 장애 상황 확인하기
# 할 일: 기본 승인 마스터가 0건이고 모든 외부 반영이 NOT_CONNECTED인지 조회하세요. 보류·후보 목록이 그대로 남는지 확인합니다. 없는 파일 경로를 읽어 수집 실패가 나는지 확인하고 기존 원파일 해시를 비교하세요. 실패한 단계를 성공으로 기록하지 않습니다. 임시 DB를 두 연결로 열어 한쪽이 잠근 동안 다른 쪽의 저장 실패를 확인하세요. 재시도가 필요하다는 처리와 기록을 남깁니다. 미션8의 실제 재시도 시험도 함께 제시합니다. 체크포인트에서 복원한 처리기에 전체 이벤트를 다시 보내고 재수신 전후 현재값·충돌 목록이 같은지 비교하세요. 시험 범위를 넘는 복구를 했다고 주장하지 않습니다.
# 확인: 기본 승인 0건, 세 장애 시험 통과를 확인하세요. 원자료 유지 확인과 모든 결과 파일의 원자적 복구는 다른 범위이므로 실제로 확인한 범위를 적습니다.
def m10_s3(c):
    """승인 없는 실행과 세 장애 상황 확인하기.

    읽을 작업값: SyncEngine, approval, candidates, checks, clean_run_ok, events, lineage_tests, load_fn, root
    저장할 작업값: e2e_tests
    결과 폴더: folder(c, 10)
    """
    return TODO(10, 3)

# STEP 04 현재 환경을 다시 평가하고 실행 매뉴얼 작성하기
# 할 일: 각 영역을 0점 근거 없음, 1점 문서·수동 확인, 2점 정상·오류를 포함한 시험 확인의 같은 기준으로 다시 평가하세요. 이전·현재 점수와 실제 시험 경로·남은 확인을 적습니다. 매뉴얼을 설치, 입력, 코드·단위, 검증·품질, 동기화·복구, 역할별 화면, 승인·감사, 부하·장애, 인수·변경의 아홉 장으로 작성하세요. 각 장에 필요한 파일·실행 또는 확인 방법·정상 결과·주요 오류와 조치를 넣으세요. 단순 파일 목록이 아니라 다음 담당자가 따라 할 순서로 씁니다. 동료에게 실행 명령 한 개와 오류 조치 한 개를 매뉴얼만 보고 찾게 하고 막힌 부분을 고치세요. 실제 환경 인증·외부 연동·최종 교관 평가는 남은 일로 구별합니다.
# 확인: 아홉 장이 모두 있고 실제 파일 경로와 명령이 맞는지 확인하세요. “오류를 해결한다” 대신 확인할 파일과 다시 실행할 조건이 있어야 합니다.
def m10_s4(c):
    """현재 환경을 다시 평가하고 실행 매뉴얼 작성하기.

    읽을 작업값: checks, clean_run_ok, environment_baseline, out
    저장할 작업값: manual_chapters
    결과 폴더: folder(c, 10)
    """
    return TODO(10, 4)

# STEP 05 최종 열두 항목을 확인하고 다음 담당자에게 인계하기
# 할 일: 12항목의 실제 근거를 열어 PASS·FAIL·NOT_RUN을 기록하세요. 자동 코드 점검과 교관 확인을 서로 다른 칸으로 유지합니다. 자동 PASS만으로 학습 합격을 표시하지 않습니다. 1쪽 브리핑에 전체·수용·보류·격리 건수, 품질 분자·분모, 값 충돌·비교 불가·승인대기, 실측 성능, 업무시간 가정, 남은 문제를 적으세요. handover.md에 다음 담당자·미완료 작업·완료 조건·확인할 파일을 적고 data·config·src·결과·로그·매뉴얼을 함께 묶으세요. 학습자 배포본과 교관 정답 자료는 분리합니다. 교관이 바꾼 거점 필터나 허용 별칭 하나에 대해 실행 전에 결과를 예상하고, 재실행 뒤 원자료와 대조해 이유를 설명하세요. 전체 필수 평가와 최종 점검을 마친 뒤 교관이 합격을 판정합니다.
# 확인: 코드 점검 12/12와 교관 평가 20/20은 다른 결과입니다. 두 종류와 최종 사람 확인이 모두 끝나기 전에는 과정 전체를 합격으로 표시하지 않습니다.
def m10_s5(c):
    """최종 열두 항목을 확인하고 다음 담당자에게 인계하기.

    읽을 작업값: approval, candidates, checks, clean_run_ok, e2e_tests, lineage_tests, load_summary, manual_chapters, normalized, population, quality, root, status_counts
    저장할 작업값: 근거 파일과 반환 표
    결과 폴더: folder(c, 10)
    """
    return TODO(10, 5)

result_html = run_mission(c, 10, [m10_s1, m10_s2, m10_s3, m10_s4, m10_s5])
document = result_html.read_text(encoding="utf-8")
display(HTML('<iframe title="DAT09 단계별 결과" style="width:100%;height:1100px;border:1px solid #ccd5dd" sandbox="allow-scripts allow-downloads allow-same-origin" srcdoc="'+html.escape(document,quote=True)+'"></iframe>'))
